# Критерий Колмогорова-Смирнова

В критерии Колмогорова мы проверяли, что наше распределение равно какому-то $F$

Здесь у нас есть две выборки, и мы хотим проверить, что они из одного распределения. При этом неважно, какого именно 

$H_0: F_X = F_Y$    
$H_1: F_X \neq F_Y$

Статистика: 
$$\sqrt{\frac{nm}{n + m}} D_{nm} \xrightarrow{p} \phi$$

$$D_{nm} = \sup_{x \in \mathbb{R}} |\widehat{F}_{X_n}(x) - \widehat{F}_{Y_m}(x)|$$

In [2]:
import numpy as np 
import pandas as pd
from scipy import stats

import matplotlib.pyplot as plt
import seaborn as sns
import random

from tqdm import tqdm

from statsmodels.graphics.gofplots import qqplot

from statsmodels.stats.proportion import proportion_confint

from collections import namedtuple

plt.rcParams['figure.dpi'] = 150

In [ ]:
# Проверяем, что критерий работает 

monteCarloResults = namedtuple('MonteCarloResults', ['type_one', 'ci_left', 'ci_right'])

def run_monte_carlo_aa(n, m, dist_x, dist_y, n_iter, alpha):
    count = 0
    for _ in range(n_iter):
        control = dist_x.rvs(n)
        test = dist_y.rvs(m)

        pvalue = stats.ktest(control, test).pvalue

        count += (pvalue <= alpha)

    type_one = count / n_iter
    ci_left, ci_right = proportion_confint(count=count, n_obs=n_iter, alpha=0.05, method='Wilson')

    return monteCarloResults(**{'type_one' : type_one, 'ci_left': ci_left, 'ci_right': ci_right})

(positive_rate, 
 confint_left_bound, 
 confint_right_bound) = run_monte_carlo_aa(n = 20, 
                                                   m = 20,
                                                   alpha = 0.05,
                                                   N_runs = 20000,
                                                   latent_dist_X = stats.gamma(5, 110),
                                                   latent_dist_Y = stats.gamma(5, 110))

print('FPR: ', positive_rate)
print('FPR confint: ', (confint_left_bound, confint_right_bound))